# 04 Apply inverse to evokeds

This notebook applies inverse operators to condition-specific evoked responses and writes source estimates (STCs).

Inputs per recording/condition:

- condition-specific evoked file from sensor analysis,
- inverse operator from `03_inverse_operator.ipynb`.

Outputs are written to `derivatives/meeg-pipeline/sub-*/meg/source_estimates/`.

## Setup

In [ ]:
from __future__ import annotations

from pathlib import Path

import mne
import pandas as pd

from meeg_pipeline.config import load_config
from meeg_pipeline.source_modeling import (
    apply_inverse_config_to_dataframe,
    apply_inverse_to_evokeds_for_recordings,
    source_estimate_input_overview_to_dataframe,
    source_estimate_qc_to_dataframe,
    source_estimate_results_to_dataframe,
)
from meeg_pipeline.workflow import (
    existing_output_policy_for_step,
    iter_recordings,
    selected_recordings_to_dataframe,
    should_overwrite,
)


def find_project_root(start: Path | None = None) -> Path:
    """Find the project root by searching upward for configs/local.yaml."""
    start = Path.cwd() if start is None else Path(start).resolve()

    for candidate in [start, *start.parents]:
        if (candidate / "configs" / "local.yaml").exists():
            return candidate

    raise FileNotFoundError(
        "Could not find project root by searching for configs/local.yaml "
        f"above {start}"
    )


PROJECT_ROOT = find_project_root()
CONFIG_PATH = PROJECT_ROOT / "configs" / "local.yaml"
config = load_config(CONFIG_PATH)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("CONFIG_PATH:", CONFIG_PATH)

## MNE logging

In [ ]:
mne.set_log_level("WARNING")

## Selection

`iter_recordings(..., "all")` uses existing raw-BIDS recordings and excludes `sub-emptyroom` by default.

In [ ]:
SUBJECTS = "all"
SESSIONS = "all"
TASKS = "all"
RUNS = "all"

RUN_SOURCE_ESTIMATE_QC = True
MAX_SOURCE_ESTIMATE_QC_ROWS = None  # None = all, or e.g. 4 for a quick test

selected_recordings = list(
    iter_recordings(
        config,
        subjects=SUBJECTS,
        sessions=SESSIONS,
        tasks=TASKS,
        runs=RUNS,
    )
)

selected_recordings_to_dataframe(selected_recordings)

## Overwrite policy

Default: skip existing source estimates.

Set `OVERWRITE_STEPS = ["source_estimate"]` to recompute existing STCs.

In [ ]:
OVERWRITE_STEPS = []

pd.DataFrame(
    [
        {
            "step": "source_estimate",
            "overwrite": should_overwrite("source_estimate", OVERWRITE_STEPS),
            "policy": existing_output_policy_for_step(
                "source_estimate",
                OVERWRITE_STEPS,
            ),
        }
    ]
)

## Apply-inverse parameters

Methodological parameters are read from `configs/local.yaml` under `source.apply_inverse`.

In [ ]:
apply_inverse_settings = apply_inverse_config_to_dataframe(config)
apply_inverse_settings

## Effective parameters

These variables are derived from the config and passed explicitly to the library helpers for clarity.

In [ ]:
APPLY_TO = config.source.apply_inverse.apply_to
METHOD = config.source.apply_inverse.method
SNR = config.source.apply_inverse.snr
LAMBDA2 = config.source.apply_inverse.lambda2
PICK_CONDITIONS = config.source.apply_inverse.pick_conditions
SAVE_STCS = config.source.apply_inverse.save_stcs

SOURCE_SPACING = config.source.spacing
NOISE_COV_MODE = config.source.noise_cov.mode

pd.DataFrame(
    [
        {
            "apply_to": APPLY_TO,
            "method": METHOD,
            "snr": SNR,
            "lambda2": LAMBDA2 if LAMBDA2 is not None else 1.0 / SNR**2,
            "pick_conditions": PICK_CONDITIONS,
            "save_stcs": SAVE_STCS,
            "source_spacing": SOURCE_SPACING,
            "noise_cov_mode": NOISE_COV_MODE,
        }
    ]
)

## Input overview

This table checks whether all required evoked files and inverse operators exist before applying the inverse.

In [ ]:
source_estimate_policy = existing_output_policy_for_step(
    "source_estimate",
    OVERWRITE_STEPS,
)

source_estimate_overview = source_estimate_input_overview_to_dataframe(
    config,
    selected_recordings,
    on_existing=source_estimate_policy,
    apply_to=APPLY_TO,
    method=METHOD,
    pick_conditions=PICK_CONDITIONS,
    spacing=SOURCE_SPACING,
    noise_cov_mode=NOISE_COV_MODE,
)

source_estimate_overview

## Status summary

In [ ]:
if source_estimate_overview.empty:
    pd.DataFrame()
else:
    (
        source_estimate_overview
        .groupby(["status"], dropna=False)
        .size()
        .reset_index(name="n_jobs")
        .sort_values(["status"])
    )

## Ready jobs

In [ ]:
ready_source_estimate_jobs = source_estimate_overview.query("status == 'ready'").copy()

columns = [
    "subject",
    "session",
    "task",
    "run",
    "condition",
    "evoked_path",
    "inverse_path",
    "stc_path",
]
existing_columns = [column for column in columns if column in ready_source_estimate_jobs.columns]
ready_source_estimate_jobs[existing_columns]

## Apply inverse to evokeds

This cell processes all selected recordings and conditions. Existing STCs are skipped unless `OVERWRITE_STEPS` contains `"source_estimate"`. Missing inputs and per-job failures are returned as status rows rather than stopping the whole batch.

In [ ]:
if APPLY_TO != "evoked":
    raise NotImplementedError(
        "This notebook currently implements apply_to='evoked'. "
        "Epoch-wise source estimates can be added later."
    )

source_estimate_results = apply_inverse_to_evokeds_for_recordings(
    config,
    selected_recordings,
    on_existing=source_estimate_policy,
    method=METHOD,
    lambda2=LAMBDA2,
    snr=SNR,
    pick_conditions=PICK_CONDITIONS,
    spacing=SOURCE_SPACING,
    noise_cov_mode=NOISE_COV_MODE,
    save_stcs=SAVE_STCS,
    verbose=True,
)

source_estimate_results_df = source_estimate_results_to_dataframe(source_estimate_results)
source_estimate_results_df

## Result summary

In [ ]:
if "source_estimate_results_df" not in globals() or source_estimate_results_df.empty:
    pd.DataFrame()
else:
    (
        source_estimate_results_df
        .groupby("status", dropna=False)
        .size()
        .reset_index(name="n_jobs")
        .sort_values(["status"])
    )

## Batch QC

Reads saved source estimates and returns a compact QC table. This is non-interactive and safe for batch runs.

In [ ]:
if RUN_SOURCE_ESTIMATE_QC:
    if "source_estimate_results_df" not in globals() or source_estimate_results_df.empty:
        source_estimate_qc_status = pd.DataFrame(
            [{"status": "no_results", "message": "Run the apply-inverse cell first."}]
        )
    else:
        candidate_results = source_estimate_results_df[
            source_estimate_results_df["status"].isin(
                ["written", "skipped_existing", "exists"]
            )
        ].copy()

        if candidate_results.empty:
            candidate_results = source_estimate_results_df.copy()

        source_estimate_qc_status = source_estimate_qc_to_dataframe(
            candidate_results,
            max_rows=MAX_SOURCE_ESTIMATE_QC_ROWS,
        )

    source_estimate_qc_status
else:
    print("Skipped source-estimate QC.")

## Expected outputs

Source estimates are written to:

```text
derivatives/meeg-pipeline/sub-*/meg/source_estimates/*_space-source_desc-*-stc.h5
```